In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.GoldHelper

In [0]:
from pyspark.sql import functions as F 

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_silver_table=f"{catalog_name}.{silver_schema}.constructors"
nationality_table=f"{catalog_name}.{gold_schema}.nationality_continent"


In [0]:
%sql
--select * from formula1_incr_catalog.gold.nationality_continent
--select * from formula1_catalog.gold.nationality_continent

create table if not exists formula1_incr_catalog.gold.nationality_continent
as
select * from formula1_catalog.gold.nationality_continent


In [0]:

df_constructions = (spark.table(constructors_silver_table)
                    .filter(F.col("batch_id") == v_batch_id)
                    )
df_nationality = spark.table(nationality_table)



In [0]:
df_constructions_dims=(
    df_constructions
    .join(df_nationality,
        df_constructions.nationality == df_nationality.Nationality,
        "left")
    .select(
        df_constructions.constructor_id.alias('constructor_id'),
        df_constructions.constructor_name.alias('constructor_name'),
        df_constructions.nationality.alias('nationality'),
        df_nationality.Continent.alias('Continent'),
        df_constructions.batch_id.alias('batch_id')
    )
)



In [0]:
display(df_constructions_dims)

In [0]:
# df_constructions_dims.write.mode("overwrite").saveAsTable(target_table)

In [0]:
write_to_gold(
    input_df=df_constructions_dims,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "Continent"
    ]
)


In [0]:
display(spark.table(target_table))